# 02 - Artificial-gap validation: running the protocol

Notebook 01 introduced the artificial-gap pool and why real gaps can't be
scored. This notebook actually **runs** the validation protocol: the three
Model 0 baselines (monthly climatology, persistence, linear interpolation)
against the released chlorophyll artificial-gap pool, scored the same way
every other method in this repository is scored. Fully executable on the
public data included in this repository.

**Scoring scale.** The final report and
`results/chlorophyll/chlorophyll_benchmark_summary.csv` score chlorophyll
on `log10(chl_mean)`, not on physical `chl_mean` (mg m^-3), because the
target distribution is strongly right-skewed. This notebook reproduces
that scale: it derives a `chl_log10` column from the public daily target
table and scores against `chl_log10`, so the MAE values below are directly
comparable to the released benchmark table. A physical-scale MAE is *not*
comparable -- it is dominated by a handful of high-chlorophyll days and is
roughly an order of magnitude larger. See `docs/methods.md` for why the
log10 transform was chosen.

In [1]:
import sys
import warnings

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from coastal_gap_reconstruction.artificial_gap_validation import apply_artificial_gap
from coastal_gap_reconstruction.baseline_imputation import run_all_baselines
from coastal_gap_reconstruction.data_loading import load_daily_target, load_validation_gap_pool
from coastal_gap_reconstruction.scoring_metrics import aggregate_metrics, compute_gap_metrics

# A handful of gaps have a single hidden day or a constant baseline
# prediction, which triggers scipy's ConstantInputWarning inside the
# correlation-coefficient calculation -- harmless (the code already
# guards the affected metric with an explicit std>0 check; this only
# silences the warning message itself), not a data or scoring problem.
warnings.filterwarnings("ignore", message="An input array is constant")

target_df = load_daily_target("../data/chlorophyll/chlorophyll_daily_target.csv")
gap_pool = load_validation_gap_pool("../data/chlorophyll/chlorophyll_validation_gaps.csv")

# Score on log10(chl_mean), matching the benchmark scale. chl_mean is
# guaranteed positive for eligible days; non-positive/NaN values map to NaN
# and are excluded downstream the same way missing values already are.
target_df["chl_log10"] = np.log10(target_df["chl_mean"].where(target_df["chl_mean"] > 0))
TARGET_COL = "chl_log10"

print(len(gap_pool), "gaps loaded")

681 gaps loaded


## Masking each gap and running the three baselines

This is the artificial-gap protocol notebook 01 described, applied for
real: for every gap in the pool, mask the target over the hidden days
(`apply_artificial_gap`), run every Model 0 baseline as if those days were
missing (`run_all_baselines`), then score each prediction against the
secretly retained true value (`compute_gap_metrics`).

In [2]:
all_metrics = []

for _, g in gap_pool.iterrows():
    start = pd.Timestamp(g["start_date"])
    gap_length = int(g["gap_length"])

    masked = apply_artificial_gap(target_df, start, gap_length, target_col=TARGET_COL)
    predictions = run_all_baselines(masked, start, gap_length, target_col=TARGET_COL)

    metrics = compute_gap_metrics(
        target_df=target_df,
        predictions=predictions,
        start_date=start,
        gap_length=gap_length,
        gap_id=g["gap_id"],
        gap_info=g.to_dict(),
        target_col=TARGET_COL,
    )
    all_metrics.extend(metrics)

metrics_df = pd.DataFrame(all_metrics)
metrics_df.head()

,gap_id,method,gap_length,season,year,target_mean_true,n_valid,coverage,mae,rmse,bias,r
0,L01_20150715,clim_monthly,1,JJA,2015,1.0391,1,1.0,0.3522,0.3522,0.3522,NaN
1,L01_20150715,persistence,1,JJA,2015,1.0391,0,0.0,NaN,NaN,NaN,NaN
2,L01_20150715,linear_interp,1,JJA,2015,1.0391,0,0.0,NaN,NaN,NaN,NaN
3,L01_20150820,clim_monthly,1,JJA,2015,2.5656,1,1.0,0.0610,0.0610,0.0610,NaN
4,L01_20150820,persistence,1,JJA,2015,2.5656,1,1.0,0.2808,0.2808,0.2808,NaN


## Aggregate by method and gap length

In [3]:
summary = aggregate_metrics(metrics_df, groupby_cols=["method", "gap_length"])
summary.sort_values(["method", "gap_length"])

,method,gap_length,n_gaps,mae_mean,mae_std,rmse_mean,rmse_std,bias_mean,r_mean,coverage_mean
0,clim_monthly,1,100,0.327031,0.257605,0.327031,0.257605,-0.002579,NaN,1.00
1,clim_monthly,3,100,0.316416,0.220669,0.332739,0.219883,0.045364,0.058671,1.00
2,clim_monthly,7,100,0.314027,0.182423,0.346323,0.191608,-0.004090,0.129035,1.00
3,clim_monthly,10,100,0.324907,0.188105,0.371880,0.198309,-0.045186,0.078290,1.00
4,clim_monthly,14,100,0.307446,0.153638,0.353583,0.159902,0.006764,0.088126,1.00
5,clim_monthly,21,80,0.318585,0.139629,0.372493,0.150369,-0.011503,0.177222,1.00
6,clim_monthly,30,50,0.320578,0.146949,0.377140,0.155075,-0.002742,0.211322,1.00
7,clim_monthly,45,29,0.329955,0.137336,0.394076,0.156096,-0.025210,0.154566,1.00
8,clim_monthly,60,22,0.315359,0.094600,0.380700,0.118602,-0.029677,0.218495,1.00
9,linear_interp,1,100,0.107697,0.097512,0.107697,0.097512,0.018919,NaN,0.99


## Aggregate by method only

In [4]:
overall = aggregate_metrics(metrics_df, groupby_cols=["method"])
overall

,method,n_gaps,mae_mean,mae_std,rmse_mean,rmse_std,bias_mean,r_mean,coverage_mean
0,clim_monthly,681,0.318656,0.187355,0.354795,0.194140,-0.003545,0.152227,1.000000
1,linear_interp,681,0.211663,0.134448,0.249641,0.162103,0.006768,0.347949,0.998532
2,persistence,681,0.284545,0.190806,0.326376,0.215785,0.018739,NaN,0.998532


## Interpretation

These three baselines establish the floor for this benchmark. Linear
interpolation typically performs best among the three for short gaps (it
has access to both edges of the gap), but is not forecast-safe. Because
these numbers are scored on `log10(chl_mean)`, they are directly
comparable to `results/chlorophyll/chlorophyll_benchmark_summary.csv`,
which scores the engineered tabular, gap-edge, and TS-ICL methods the same
way (notebooks 03 and 04). Exact values will differ slightly from the
released table because the released benchmark applies additional
bootstrap/day-weighting and stratum matching described in the report; this
notebook reproduces the underlying per-gap MAE, not the exact published
aggregation.